# SER - Experiment 09: export the sweep winner for the local demo

Notebook 06 saved every candidate model but not the per-stream scalers, which
were fitted in memory. The demo needs both.

This refits the winner's scalers from the cached 3x training features and
exports the model weights and scaler statistics as plain numpy arrays, so the
local Keras 2 environment can load them (a Keras 3 `.keras` file cannot be
read by Keras 2).

**Attach:** the four corpora, `ser-feature-cache`, **and notebook 06's
output**.
**Accelerator: GPU or CPU.** About 3 minutes.

In [ ]:
import glob
import json
import os
import shutil
import sys

import numpy as np
import tensorflow as tf
import keras

print("TF", tf.__version__, "| Keras", keras.__version__)

REPO = "https://github.com/Eldorado5002/ser.git"
if not os.path.exists("/kaggle/working/ser"):
    !git clone -q {REPO} /kaggle/working/ser
sys.path.insert(0, "/kaggle/working/ser")
os.chdir("/kaggle/working/ser")
!git log --oneline -1

In [ ]:
DATA_ROOT = "/kaggle/working/ser/data"
CANONICAL = {
    "RAVDESS": "audio_speech_actors_01-24",
    "TESS":    "TESS Toronto emotional speech set data",
    "SAVEE":   "ALL",
    "CREMA-D": "AudioWAV",
}


def find_canonical(target):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        for d in dirs:
            if d.lower() == target.lower():
                hits.append(os.path.join(root, d))
    return sorted(hits)[0] if hits else None


os.makedirs(DATA_ROOT, exist_ok=True)
for name, target in CANONICAL.items():
    src = find_canonical(target)
    assert src is not None, f"MISSING INPUT for {name}"
    dst = os.path.join(DATA_ROOT, name)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)

import config
from data_loader import build_metadata, split_metadata
from augmentation import plan_augmentation
from features import build_feature_matrix
from utils import StreamScalers, set_seed

config.CACHE_DIR = "/kaggle/working/features_cache"
os.makedirs(config.CACHE_DIR, exist_ok=True)
staged = 0
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".npz"):
            shutil.copy(os.path.join(root, f), config.CACHE_DIR)
            staged += 1
print(f"staged {staged} cache files")

meta = build_metadata(strict=True)
train_df, val_df, test_df = split_metadata(meta)
set_seed(config.RANDOM_SEED)

# The winner used 3x augmentation. Refit its scalers exactly as notebook 06
# did: same deterministic plan, same cache, therefore identical statistics.
orig = config.TARGET_TRAIN_SIZE
config.TARGET_TRAIN_SIZE = 3 * len(train_df)
items = plan_augmentation(train_df, emotion_aware=False)
config.TARGET_TRAIN_SIZE = orig
train_3x = build_feature_matrix(items, desc="train_uniform3x")
print("train rows:", train_3x["mfcc"].shape[0])

scalers = StreamScalers().fit(train_3x)

In [ ]:
from model import build_model

TAG = "gap_reg_aug3"
BUILD_KW = dict(use_afw=False, use_mstc=False, head="gap",
                dropout_conv=0.35, dropout_dense=0.55, l2=1e-4)

src = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == f"sweep_{TAG}.keras":
            src = os.path.join(root, f)
assert src, f"sweep_{TAG}.keras not found - attach notebook 06's output"
print("model:", src)

loaded = tf.keras.models.load_model(src, compile=False)
w = loaded.get_weights()

# The demo rebuilds the architecture from model.py, so verify shapes match.
fresh, _ = build_model(**BUILD_KW)
fw = fresh.get_weights()
assert len(w) == len(fw), f"{len(w)} arrays vs {len(fw)}"
for i, (a, b) in enumerate(zip(w, fw)):
    assert a.shape == b.shape, f"array {i}: {a.shape} vs {b.shape}"
print(f"{len(w)} weight arrays, {loaded.count_params():,} params - shapes OK")

OUT = "/kaggle/working/demo_export"
os.makedirs(OUT, exist_ok=True)

np.savez_compressed(os.path.join(OUT, "best.weights.npz"), *w)

sc = {}
for name in StreamScalers.STREAMS:
    s = scalers.scalers[name]
    sc[f"{name}_mean"] = s.mean_.astype(np.float32)
    sc[f"{name}_scale"] = s.scale_.astype(np.float32)
np.savez_compressed(os.path.join(OUT, "best.scalers.npz"), **sc)

# A reference prediction so the local Keras 2 side can verify the round trip.
r = np.random.default_rng(0)
x = [r.standard_normal((3, config.MFCC_LEN)).astype("float32"),
     r.standard_normal((3, config.ZCR_LEN)).astype("float32"),
     r.standard_normal((3, config.RMSE_LEN)).astype("float32")]
np.savez(os.path.join(OUT, "best.reference.npz"),
         mfcc=x[0], zcr=x[1], rmse=x[2], pred=loaded.predict(x, verbose=0))

json.dump({"tag": TAG, "build_kw": {k: str(v) for k, v in BUILD_KW.items()},
           "params": int(loaded.count_params())},
          open(os.path.join(OUT, "best.config.json"), "w"), indent=2)

for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    print(f"  {os.path.getsize(p)/1e6:7.2f} MB  {f}")

for name in CANONICAL:
    link = os.path.join(DATA_ROOT, name)
    if os.path.islink(link):
        os.unlink(link)
print("\ndone - download the demo_export folder")